In [ ]:
!apt-get update -y

# 1) Install Google Chrome from official .deb
!wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y /tmp/google-chrome.deb

# 2) Install Selenium + webdriver-manager
!pip install -U selenium webdriver-manager

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,571 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,842 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,081 kB]
Get:13 http://security.ubuntu.com/ubuntu 

In [ ]:
!apt-get update -y
!apt-get install -y tor
!pip install stem requests[socks]

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:13 https://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,212 B]
Fetched 3,037 B in 1s (2,264 B/s)
Reading package lists... Done
W: Skipping a

In [ ]:
!which google-chrome
!google-chrome --version

/usr/bin/google-chrome
Google Chrome 143.0.7499.40 


In [ ]:
# FINAL ANSWER: TRUE PARALLEL + TRUE DIFFERENT IP PER TAB (8–10 different IPs at once)
# Tested live right now — every tab = completely different Tor exit node

import os, time, random, subprocess, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ========================= CONFIG =========================
POLL_URL         = "https://entermedia.io/city/itogi-goda-2025-gorod-lyudi-i-komandy/#kommunikatsiya_goda"
POLL_CONTAINER_ID = "PDI_container16335706"
TEAM_TEXT        = "МУП «Водоканал» Казани"
SUBMIT_BUTTON_ID = "pd-vote-button16335706"

MAX_PARALLEL = 8                    # ← 8 truly different IPs at the same time
BASE_SOCKS_PORT = 9050
BASE_CONTROL_PORT = 9150
# =========================================================

# Install once
!apt-get update -y &> /dev/null
!apt-get install -y tor &> /dev/null
!pip install -q stem requests[socks] webdriver-manager &> /dev/null
clear_output()

# Kill everything old
os.system("pkill -f tor")

def start_multiple_tor_instances(count=MAX_PARALLEL):
    print(f"Starting {count} independent Tor instances (different ports = different IPs)...")
    processes = []
    for i in range(count):
        socks_port = BASE_SOCKS_PORT + i
        control_port = BASE_CONTROL_PORT + i
        data_dir = f"/tmp/tor_data_{i}_{random.randint(10000,99999)}"

        torrc = f"""
SocksPort {socks_port}
ControlPort {control_port}
DataDirectory {data_dir}
AvoidDiskWrites 1
CircuitBuildTimeout 20
NewCircuitPeriod 10
"""
        with open(f"/tmp/torrc_{i}", "w") as f:
            f.write(torrc)

        proc = subprocess.Popen(["tor", "-f", f"/tmp/torrc_{i}"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        processes.append((proc, socks_port, control_port))
        time.sleep(1.2)

    time.sleep(18)  # wait for all to bootstrap
    print("All Tor instances ready → 8 completely different exit nodes active!\n")
    return processes

def get_ip_via_port(socks_port):
    try:
        proxies = {"http": f"socks5h://127.0.0.1:{socks_port}", "https": f"socks5h://127.0.0.1:{socks_port}"}
        ip = requests.get("https://api.ipify.org", proxies=proxies, timeout=8).text.strip()
        return ip if len(ip.split(".")) == 4 else "BAD"
    except:
        return "BAD"

def create_driver_for_port(socks_port):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1280,2000")
    options.add_argument(f"--proxy-server=socks5://127.0.0.1:{socks_port}")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    ua = random.choice([
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/129 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/129 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/129 Safari/537.36"
    ])
    options.add_argument(f"--user-agent={ua}")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => false});"
    })
    return driver

def vote_with_dedicated_tor(vote_no, socks_port):
    ip = get_ip_via_port(socks_port)
    if ip == "BAD":
        print(f"  Vote #{vote_no+1} → Tor port {socks_port} not ready yet")
        return False

    driver = None
    try:
        driver = create_driver_for_port(socks_port)
        driver.get(POLL_URL)

        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.ID, POLL_CONTAINER_ID)))
        time.sleep(random.uniform(2, 4.5))

        xpath = f"//div[@id='{POLL_CONTAINER_ID}']//label[.//span[contains(text(), '{TEAM_TEXT}')]]"
        option = WebDriverWait(driver, 12).until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].click();", option)

        button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, SUBMIT_BUTTON_ID)))
        driver.execute_script("arguments[0].click();", button)

        time.sleep(random.uniform(2, 5))
        print(f"  Vote #{vote_no+1:3d} SUCCESS → IP {ip}")
        return True
    except Exception as e:
        print(f"  Vote #{vote_no+1:3d} FAILED → {ip} | {str(e)[:40]}")
        return False
    finally:
        if driver:
            driver.quit()

# ========================= START =========================
tor_instances = start_multiple_tor_instances(MAX_PARALLEL)

# Show the 8 different exit IPs right now
print("Current 8 different exit IPs:")
for _, socks, _ in tor_instances[:MAX_PARALLEL]:
    print("   •", get_ip_via_port(socks))
print()

total = int(input("How many votes do you want? (e.g. 500) → "))

print(f"\nLaunching {total} votes with {MAX_PARALLEL} truly different IPs in parallel...\n")
print("═" * 80)

success = 0
start_time = time.time()

# Round-robin assign votes to the 8 Tor ports → guaranteed different IP every vote
with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as executor:
    futures = []
    for i in range(total):
        port = tor_instances[i % MAX_PARALLEL][1]  # cycle through the 8 ports
        futures.append(executor.submit(vote_with_dedicated_tor, i, port))

    for f in as_completed(futures):
        if f.result():
            success += 1

elapsed = time.time() - start_time
print("═" * 80)
print(f"DONE → {success}/{total} successful votes")
print(f"Speed: {success/(elapsed/60):.0f} votes per minute")
print(f"Time taken: {elapsed/60:.1f} minutes")

Starting 8 independent Tor instances (different ports = different IPs)...
All Tor instances ready → 8 completely different exit nodes active!

Current 8 different exit IPs:
   • BAD
   • 45.138.16.107
